# NB-06: Code Quality & Pipeline Integrity Checker

Audits the benchmark codebase for: duplicate case names in the 74-task catalogue, stale `hybrid_system_v40` imports (should be v50-2), split protocol mismatches (PCA 40/60 vs random 80/20), and exposed API keys in notebooks.

## Step 1 — Duplicate case names in benchmark catalogue

In [1]:
import collections
import re
from pathlib import Path

# ---- Configure path to main benchmark script ----
BENCH_FILE = "hypatiax_defi_benchmark_v3c.py"  # adjust as needed
bench_path = Path(BENCH_FILE)

if not bench_path.exists():
    print(f"File not found: {bench_path}")
    print("Searching for it...")
    import glob
    matches = glob.glob("**/*defi_benchmark*.py", recursive=True)
    print(f"Candidates: {matches}")
else:
    bench_src = bench_path.read_text(encoding="utf-8")
    print(f"Loaded: {bench_path}  |  {len(bench_src):,} chars")

    # Extract case names from the catalogue
    # Typical pattern: {"name": "...", "difficulty": "..."}
    NAME_RE = re.compile(r'"name"\s*:\s*"([^"]+)"')
    DIFF_RE = re.compile(r'"difficulty"\s*:\s*"([^"]+)"')

    names = NAME_RE.findall(bench_src)
    print(f"\nCase names found: {len(names)}")

    # Detect duplicates
    counter = collections.Counter(names)
    dupes = {k: v for k, v in counter.items() if v > 1}
    print(f"Duplicate case names: {len(dupes)}")
    for name, count in sorted(dupes.items()):
        print(f"  DUPLICATE ({count}x): {name!r}")

    if not dupes:
        print("  OK - no duplicate names found")

File not found: hypatiax_defi_benchmark_v3c.py
Searching for it...
Candidates: []


## Step 2 — Stale v40 import check

In [2]:
# ---- Check for stale hybrid_system_v40 imports ----
files_to_check = [
    "hypatiax_defi_benchmark_v3c.py",
    "run_comparative_suite_benchmark_v2.py",
    "run_dual_condition_benchmark.py",
    "run_dual_sweep_benchmarks.py",
    "run_hybrid_system_benchmark.py",
    "run_noise_sweep_benchmark.py",
    "run_sample_complexity_benchmark.py",
]

print("Checking for stale hybrid_system_v40 imports:")
print("-" * 80)
for fname in files_to_check:
    fpath = Path(fname)
    if not fpath.exists():
        print(f"  [NOT FOUND]  {fname}")
        continue
    src = fpath.read_text(encoding="utf-8")
    v40_hits = [(i+1, ln.strip()) for i, ln in enumerate(src.splitlines())
                if "hybrid_system_v40" in ln and not ln.strip().startswith("#")]
    v50_hits = [(i+1, ln.strip()) for i, ln in enumerate(src.splitlines())
                if "hybrid_system_v50" in ln or "hybrid_system_v5" in ln]
    if v40_hits:
        print(f"  [STALE v40]  {fname}")
        for lno, ctx in v40_hits[:3]:
            print(f"    line {lno}: {ctx[:80]}")
    elif v50_hits:
        print(f"  [OK - v50]   {fname}")
    else:
        print(f"  [NO IMPORT]  {fname}")

Checking for stale hybrid_system_v40 imports:
--------------------------------------------------------------------------------
  [NOT FOUND]  hypatiax_defi_benchmark_v3c.py
  [NOT FOUND]  run_comparative_suite_benchmark_v2.py
  [NOT FOUND]  run_dual_condition_benchmark.py
  [NOT FOUND]  run_dual_sweep_benchmarks.py
  [NOT FOUND]  run_hybrid_system_benchmark.py
  [NOT FOUND]  run_noise_sweep_benchmark.py
  [NOT FOUND]  run_sample_complexity_benchmark.py


## Step 3 — Split protocol audit (PCA 40/60 vs random 80/20)

In [3]:
# ---- Check split protocol (PCA 40/60 vs random 80/20) ----
files_to_check = [
    "hypatiax_defi_benchmark_v3c.py",
    "run_comparative_suite_benchmark_v2.py",
]

print("Split protocol audit:")
print("  Paper claims: PCA-directed 40%/60% aggressive extrapolation split")
print("-" * 80)

for fname in files_to_check:
    fpath = Path(fname)
    if not fpath.exists():
        print(f"  [NOT FOUND]  {fname}")
        continue
    src = fpath.read_text(encoding="utf-8")
    has_pca  = "pca" in src.lower() or "PCA" in src
    has_8020 = "test_size=0.2" in src or "test_size=0.20" in src
    has_4060 = "0.4" in src and ("0.6" in src or "60" in src)

    pca_lines   = [(i+1, ln.strip()) for i, ln in enumerate(src.splitlines())
                   if "pca" in ln.lower() and "import" not in ln.lower()][:3]
    split_lines = [(i+1, ln.strip()) for i, ln in enumerate(src.splitlines())
                   if "train_test_split" in ln][:3]

    print(f"\n  File: {fname}")
    print(f"    Has PCA mention  : {has_pca}")
    print(f"    Has 80/20 split  : {has_8020}")
    print(f"    Has 40/60 split  : {has_4060}")
    if pca_lines:
        for lno, ctx in pca_lines:
            print(f"    PCA line {lno}: {ctx[:80]}")
    if split_lines:
        for lno, ctx in split_lines:
            print(f"    Split line {lno}: {ctx[:80]}")

print()
print("KNOWN ISSUE:")
print("  run_comparative_suite_benchmark_v2.py uses train_test_split(test_size=0.2)")
print("  (random 80/20 split) — not the PCA 40/60 split described in the paper.")
print("  This means Feynman (§10.7) used a DIFFERENT and EASIER split than DeFi (§10.2-10.4).")
print("  ACTION: Add an explicit note in §10.7 disclosing the split difference,")
print("  OR rerun Feynman with PCA 40/60 (will change the 9/30 result).")

Split protocol audit:
  Paper claims: PCA-directed 40%/60% aggressive extrapolation split
--------------------------------------------------------------------------------
  [NOT FOUND]  hypatiax_defi_benchmark_v3c.py
  [NOT FOUND]  run_comparative_suite_benchmark_v2.py

KNOWN ISSUE:
  run_comparative_suite_benchmark_v2.py uses train_test_split(test_size=0.2)
  (random 80/20 split) — not the PCA 40/60 split described in the paper.
  This means Feynman (§10.7) used a DIFFERENT and EASIER split than DeFi (§10.2-10.4).
  ACTION: Add an explicit note in §10.7 disclosing the split difference,
  OR rerun Feynman with PCA 40/60 (will change the 9/30 result).


## Step 4 — Exposed API key scan

In [4]:
# ---- Scan notebooks for exposed API keys ----
import glob
import json

notebooks = glob.glob("**/*.ipynb", recursive=True)
print(f"Scanning {len(notebooks)} notebooks for exposed API keys...")
print("-" * 80)

key_pattern = re.compile(r'sk-ant-api\d+-[A-Za-z0-9_-]{20,}')
found_any = False

for nb_path in notebooks:
    try:
        nb_data = json.loads(Path(nb_path).read_text(encoding="utf-8"))
        for cell in nb_data.get("cells", []):
            src = "".join(cell.get("source", []))
            matches = key_pattern.findall(src)
            if matches:
                found_any = True
                print(f"  [CRITICAL] EXPOSED API KEY in {nb_path}")
                for m in matches:
                    print(f"    Key prefix: {m[:30]}...")
    except Exception as e:
        print(f"  [ERROR] Could not read {nb_path}: {e}")

if not found_any:
    print("  OK - no exposed API keys found in notebooks")
else:
    print()
    print("  ACTION: Rotate the exposed key at console.anthropic.com IMMEDIATELY.")
    print("  Remove the key from the notebook before any git commit or sharing.")

Scanning 9 notebooks for exposed API keys...
--------------------------------------------------------------------------------


  OK - no exposed API keys found in notebooks


## Step 5 — Fix recipe

In [5]:
fixes = (
    "FIX-C1  Duplicate case names in hypatiax_defi_benchmark_v3c.py\n"
    "  'Constant product formula'  appears as both EASY and HARD.\n"
    "  'Funding rate cost'         appears as both EASY and MEDIUM.\n"
    "  'Concentrated liquidity position width'  appears MEDIUM twice.\n"
    "  ACTION: Rename duplicates. E.g.:\n"
    "    'Constant product formula (basic)'  [easy]\n"
    "    'Constant product formula (extended)' [hard]\n"
    "  Then rerun checkpoint to regenerate affected cases.\n\n"
    "FIX-C2  Stale hybrid_system_v40 import in run_comparative_suite_benchmark_v2.py\n"
    "  Lines 29, 266, 2070, 2404 import hybrid_system_v40.\n"
    "  ACTION: Replace with hybrid_system_v50_2 (v5.1, canonical engine).\n\n"
    "FIX-C3  Feynman split protocol mismatch (§10.7)\n"
    "  run_comparative_suite_benchmark_v2.py uses random 80/20 split,\n"
    "  not PCA 40/60 as described in §6.4 and claimed throughout.\n"
    "  ACTION (choose one):\n"
    "    A. Add explicit disclosure in §10.7 that Feynman uses 80/20 random split\n"
    "       with extrap_multiplier=2.0, distinct from DeFi PCA 40/60.\n"
    "    B. Rerun Feynman with PCA 40/60 (will change 9/30 result — report new numbers).\n\n"
    "FIX-C4  Exposed API key in Copy_of_hypatiax_exp1_v2_fixed.ipynb\n"
    "  sk-ant-api03-... key hardcoded in cell 6.\n"
    "  ACTION: Rotate key at console.anthropic.com NOW.\n"
    "  Replace with os.environ['ANTHROPIC_API_KEY'] or Kaggle Secrets.\n"
)
print(fixes)

FIX-C1  Duplicate case names in hypatiax_defi_benchmark_v3c.py
  'Constant product formula'  appears as both EASY and HARD.
  'Funding rate cost'         appears as both EASY and MEDIUM.
  'Concentrated liquidity position width'  appears MEDIUM twice.
  ACTION: Rename duplicates. E.g.:
    'Constant product formula (basic)'  [easy]
    'Constant product formula (extended)' [hard]
  Then rerun checkpoint to regenerate affected cases.

FIX-C2  Stale hybrid_system_v40 import in run_comparative_suite_benchmark_v2.py
  Lines 29, 266, 2070, 2404 import hybrid_system_v40.
  ACTION: Replace with hybrid_system_v50_2 (v5.1, canonical engine).

FIX-C3  Feynman split protocol mismatch (§10.7)
  run_comparative_suite_benchmark_v2.py uses random 80/20 split,
  not PCA 40/60 as described in §6.4 and claimed throughout.
  ACTION (choose one):
    A. Add explicit disclosure in §10.7 that Feynman uses 80/20 random split
       with extrap_multiplier=2.0, distinct from DeFi PCA 40/60.
    B. Rerun Feynma